# Gene (RNA-seq) Data Preprocessing

This notebook prepares the **transcriptomic (RNA-seq) inputs** and the **aligned survival labels** that SlotSPE consumes. Starting from the raw files downloaded from [cBioPortal](https://www.cbioportal.org/), it produces, for a given TCGA study:

- `raw_rna_data_inter/<study>_rna_inter.csv` — a cleaned gene × patient expression matrix
- `clinical/all/<study>.csv` — per-patient survival endpoints, censorship flags, and matched WSI slide IDs
- `splits/5fold/<study>/fold_<k>.csv` — 5-fold cross-validation case splits

> **Note:** `TCGA-BRCA` (breast cancer) is used as the running example. The same workflow applies to any TCGA study by changing the `study` variable.

## 1. Download the data from cBioPortal

Download both the genomic data and the clinical table from the [study summary page](https://www.cbioportal.org/study/summary?id=brca_tcga_pan_can_atlas_2018). This yields:

- the study folder `brca_tcga_pan_can_atlas_2018/`, which contains `data_mrna_seq_v2_rsem_zscores_ref_all_samples.txt` (RNA-seq RSEM z-scores)
- the clinical table `brca_tcga_pan_can_atlas_2018_clinical_data.tsv`

<img src="gene_download.png" width="100%" align='left' />

## 2. Reorganize the RNA-seq data

Load the raw RNA-seq z-score matrix (genes × samples) and clean it so it can be indexed by gene symbol and matched to patients:

- drop the `Entrez_Gene_Id` column and rows with a missing `Hugo_Symbol`
- use the gene symbol as the row index
- trim the trailing sample-type code (e.g. `-01`) from the column names so they become patient-level case IDs
- drop duplicated genes, keeping the first occurrence

In [26]:
from utils import load_tsv_txt_data, reorganize_rna_seq_data
import os
root_dir = "/Data/Pathology"
study = "brca"

# load the raw RNA-seq z-score matrix (genes x samples) downloaded from cBioPortal
print("\n step 1: reorganize the rna data\n")
rna_seq_path = f"{root_dir}/RNA/{study}_tcga/data_mrna_seq_v2_rsem_zscores_ref_all_samples.txt"
rna_data = load_tsv_txt_data(rna_seq_path)
rna_data


 step 1: reorganize the rna data



,Hugo_Symbol,Entrez_Gene_Id,TCGA-3C-AAAU-01,TCGA-3C-AALI-01,TCGA-3C-AALJ-01,TCGA-3C-AALK-01,TCGA-4H-AAAK-01,TCGA-5L-AAT0-01,TCGA-5T-A9QA-01,TCGA-A1-A0SB-01,...,TCGA-UL-AAZ6-01,TCGA-UU-A93S-01,TCGA-V7-A7HQ-01,TCGA-W8-A86G-01,TCGA-WT-AB41-01,TCGA-WT-AB44-01,TCGA-XX-A899-01,TCGA-XX-A89A-01,TCGA-Z7-A8R5-01,TCGA-Z7-A8R6-01
0,NaN,100130426,-1.7608,-1.7608,1.1240,-1.7608,-1.7608,-1.7608,-1.7608,-1.7608,...,-1.7608,-1.7608,-1.7608,-1.7608,-1.7608,-1.7608,-1.7608,-1.7608,-1.7608,-1.7608
1,NaN,100133144,1.0923,0.3525,0.6434,0.6945,-0.0258,-0.6632,-1.6559,1.0067,...,-2.4527,-0.5747,-2.9255,-0.2774,-2.9255,-2.9255,0.9220,1.5076,-1.2605,-0.0187
2,UBE2Q2P2,100134869,1.0262,1.4779,0.5227,0.7937,1.1927,1.1036,0.4850,1.1889,...,1.1866,0.7669,-0.9761,1.1865,-0.0253,-0.9062,1.8034,2.1801,0.1806,2.0295
3,HMGB1P1,10357,-1.7019,-1.0047,0.9112,0.7417,-0.5515,0.1845,0.0387,0.2291,...,0.5990,0.6080,-3.0160,0.4650,-1.6351,-2.2035,0.6370,-1.2061,-0.5278,1.1108
4,NaN,10431,-2.5301,-1.6372,0.7974,-0.4534,-0.7896,-0.6444,0.9160,-1.2139,...,1.1929,2.8766,1.4480,-1.0492,0.5520,1.6871,-1.2330,-0.9517,0.3085,-0.0445
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20526,ZYG11A,440590,1.1724,0.9329,1.3990,0.8211,-1.4208,-1.3181,-2.8614,-1.1447,...,0.0404,-1.5838,-0.5071,0.4193,-0.3732,0.3000,-1.0639,0.7622,-0.2746,0.6140
20527,ZYG11B,79699,1.0090,-0.9114,-1.2595,-0.8941,-0.2173,-0.8084,-2.6508,1.3455,...,-1.5181,-1.1219,-2.1861,-1.2504,-3.5541,-2.7066,-0.5486,-0.3225,-1.3994,-1.0529
20528,ZYX,7791,0.0068,0.9300,0.9129,0.9983,0.2973,0.3426,-0.1670,1.1693,...,0.3581,0.9630,1.2557,0.9470,1.3251,2.3362,0.7809,0.9199,1.3250,-0.0552
20529,ZZEF1,23140,1.2495,0.3581,-0.4664,-0.9288,-0.7740,-0.5210,-0.0542,1.2962,...,0.2116,-1.8431,-1.0365,-0.3964,-1.7535,-0.8681,1.2995,0.5701,-1.0143,-1.7270


In [27]:
# clean up: gene-symbol index, patient-level columns, deduplicated genes
rna_data = reorganize_rna_seq_data(rna_data)
# save the reorganized expression matrix (genes x patients)
os.makedirs(f"./dataset_csv_example/raw_rna_data_inter", exist_ok=True)
rna_data.to_csv(f"./dataset_csv_example/raw_rna_data_inter/{study}_rna_inter.csv")
rna_data

,TCGA-3C-AAAU,TCGA-3C-AALI,TCGA-3C-AALJ,TCGA-3C-AALK,TCGA-4H-AAAK,TCGA-5L-AAT0,TCGA-5T-A9QA,TCGA-A1-A0SB,TCGA-A1-A0SD,TCGA-A1-A0SE,...,TCGA-UL-AAZ6,TCGA-UU-A93S,TCGA-V7-A7HQ,TCGA-W8-A86G,TCGA-WT-AB41,TCGA-WT-AB44,TCGA-XX-A899,TCGA-XX-A89A,TCGA-Z7-A8R5,TCGA-Z7-A8R6
UBE2Q2P2,1.0262,1.4779,0.5227,0.7937,1.1927,1.1036,0.4850,1.1889,0.8264,-0.5060,...,1.1866,0.7669,-0.9761,1.1865,-0.0253,-0.9062,1.8034,2.1801,0.1806,2.0295
HMGB1P1,-1.7019,-1.0047,0.9112,0.7417,-0.5515,0.1845,0.0387,0.2291,-1.3557,0.8930,...,0.5990,0.6080,-3.0160,0.4650,-1.6351,-2.2035,0.6370,-1.2061,-0.5278,1.1108
RNU12-2P,-1.9713,0.6577,-1.9713,-0.7346,-0.7049,-1.9713,-0.4685,-0.6428,-0.9504,-1.9713,...,-1.9713,-1.9713,-0.2572,-0.5839,-1.9713,3.8294,2.7559,-0.5892,-0.2984,-0.9046
SSX9P,-1.0279,-0.5659,-1.0279,-1.0279,-1.0279,-1.0279,-1.0279,-1.0279,-1.0279,-1.0279,...,-0.4035,-1.0279,-1.0279,-1.0279,-1.0279,-1.0279,-1.0279,-1.0279,-1.0279,-1.0279
EZHIP,0.0599,3.6512,-0.8451,-0.8451,-0.5249,-0.8451,-0.8451,-0.2648,-0.3864,-0.8451,...,-0.8451,2.0316,-0.8451,-0.2422,0.7322,-0.3069,0.3503,1.7895,-0.8451,-0.5754
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZYG11A,1.1724,0.9329,1.3990,0.8211,-1.4208,-1.3181,-2.8614,-1.1447,-0.6760,-0.1827,...,0.0404,-1.5838,-0.5071,0.4193,-0.3732,0.3000,-1.0639,0.7622,-0.2746,0.6140
ZYG11B,1.0090,-0.9114,-1.2595,-0.8941,-0.2173,-0.8084,-2.6508,1.3455,0.0938,0.3740,...,-1.5181,-1.1219,-2.1861,-1.2504,-3.5541,-2.7066,-0.5486,-0.3225,-1.3994,-1.0529
ZYX,0.0068,0.9300,0.9129,0.9983,0.2973,0.3426,-0.1670,1.1693,0.0372,-0.3077,...,0.3581,0.9630,1.2557,0.9470,1.3251,2.3362,0.7809,0.9199,1.3250,-0.0552
ZZEF1,1.2495,0.3581,-0.4664,-0.9288,-0.7740,-0.5210,-0.0542,1.2962,0.2830,-0.5098,...,0.2116,-1.8431,-1.0365,-0.3964,-1.7535,-0.8681,1.2995,0.5701,-1.0143,-1.7270


## 3. Process the clinical / survival data

Build the per-patient label table that pairs survival endpoints with whole-slide image (WSI) IDs, intersect it with the RNA-seq samples, and create cross-validation splits.

For each of the four TCGA survival endpoints we keep the survival time and a **censorship flag**, where `1 = censored` (event not observed) and `0 = uncensored` (event observed):

- `dfs` — Disease-Free Survival
- `dss` — Disease-Specific Survival
- `os` — Overall Survival
- `pfs` — Progression-Free Survival

In [28]:
from utils import load_tsv_txt_data, get_clinical_label, get_intersection_between_rna_and_clinical, load_csv_data

# 3.1 (optional) reorganize the WSI data into a single flat folder.
#     Skip this if your slides are already collected under one directory.
# data_dir   = f"/Data/Pathology/Slides/{study}_original/"
# target_dir = f"/Data/Pathology/Slides/{study}/"
# reorganize_wsi_data(data_dir, target_dir)   # -> target_dir/*.svs

# 3.2 build the clinical label table
print("\n step 3.2: get the clinical data \n")
slides_path = f"/Data/Pathology/Slides/{study}/"
# slides_path = f"{root_dir}/UNI/{study}/pt_files/"
status = ["Disease Free Status", "Disease-specific Survival status", "Overall Survival Status", "Progression Free Status"]
clinical_path = f"{root_dir}/labels/survival/{study}/{study}_tcga_pan_can_atlas_2018_clinical_data.tsv"
clinical = load_tsv_txt_data(clinical_path)
# map raw status strings to censorship flags: censored -> 1, event observed -> 0
# (e.g. OS: 0:LIVING -> 1, 1:DECEASED -> 0); only patients with a matching slide are kept
clinical = get_clinical_label(clinical, slides_path, status)
os.makedirs("./dataset_csv_example/clinical/all", exist_ok=True)
clinical.to_csv(f"./dataset_csv_example/clinical/all/{study}.csv")
# count censored vs. uncensored samples for the overall-survival endpoint
censored = clinical['censorship_os'].sum()
uncensored = len(clinical) - censored
print(f"Number of censored samples: {censored}")
print(f"Number of uncensored samples: {uncensored}")
clinical



 step 3.2: get the clinical data 

the numbers of Disease Free Status: 
 Disease Free Status
0:DiseaseFree            858
1:Recurred/Progressed     84
Name: count, dtype: int64
the numbers of Disease-specific Survival status: 
 Disease-specific Survival status
0:ALIVE OR DEAD TUMOR FREE    981
1:DEAD WITH TUMOR              83
Name: count, dtype: int64
the numbers of Overall Survival Status: 
 Overall Survival Status
0:LIVING      933
1:DECEASED    151
Name: count, dtype: int64
the numbers of Progression Free Status: 
 Progression Free Status
0:CENSORED       938
1:PROGRESSION    145
Name: count, dtype: int64
Number of censored samples: 902
Number of uncensored samples: 146


,case id,survival_months_dfs,censorship_dfs,survival_months_dss,censorship_dss,survival_months_os,censorship_os,survival_months_pfs,censorship_pfs,wsi
1,TCGA-3C-AALI,131.669790,1,131.669790,1,131.6697899,1,131.669790,1,TCGA-3C-AALI-01Z-00-DX2.CF4496E0-AB52-4F3E-BDF...
2,TCGA-3C-AALJ,48.459743,1,48.459743,1,48.45974291,1,48.459743,1,TCGA-3C-AALJ-01Z-00-DX1.777C0957-255A-42F0-9EE...
3,TCGA-3C-AALK,NaN,NaN,47.604958,1,47.60495775,1,47.604958,1,TCGA-3C-AALK-01Z-00-DX1.4E6EB156-BB19-410F-878...
4,TCGA-4H-AAAK,11.440971,1,11.440971,1,11.44097051,1,11.440971,1,TCGA-4H-AAAK-01Z-00-DX1.ABF1B042-1970-4E28-867...
5,TCGA-5L-AAT0,NaN,NaN,48.558372,1,48.55837196,1,48.558372,1,TCGA-5L-AAT0-01Z-00-DX1.5E171263-30BF-4C6B-88A...
...,...,...,...,...,...,...,...,...,...,...
1079,TCGA-WT-AB44,29.029819,1,29.029819,1,29.02981885,1,29.029819,1,TCGA-WT-AB44-01Z-00-DX1.B6ECEA7C-DA26-4B34-88C...
1080,TCGA-XX-A899,15.353256,1,15.353256,1,15.3532564,1,15.353256,1,TCGA-XX-A899-01Z-00-DX1.08FE27B7-73B8-4CE3-ACF...
1081,TCGA-XX-A89A,16.043660,1,16.043660,1,16.0436598,1,16.043660,1,TCGA-XX-A89A-01Z-00-DX1.671E2AD6-4D1A-4579-88C...
1082,TCGA-Z7-A8R5,NaN,NaN,108.064569,1,108.0645692,1,5.950620,0,TCGA-Z7-A8R5-01Z-00-DX1.3BDB407F-514C-4131-B05...


In [29]:
# 3.3 keep only patients present in BOTH the RNA-seq matrix and the clinical table
from utils import load_csv_data
print("\n step 3.3: intersection between rna and clinical \n")
print("the numbers of clinical: ", clinical.shape[0])
print("the numbers of rna_data: ", rna_data.shape[1])

clinical_new = get_intersection_between_rna_and_clinical(rna_data, clinical)
print("the numbers of clinical_new: ", clinical_new.shape[0])
# reset the index and overwrite the clinical table with the intersected cohort
clinical_new = clinical_new.reset_index(drop=True)
clinical_new.to_csv(f"./dataset_csv_example/clinical/all/{study}.csv")


 step 3.3: intersection between rna and clinical 

the numbers of clinical:  1048
the numbers of rna_data:  1082
the numbers of clinical_new:  1046


In [31]:
# 3.4 generate 5-fold cross-validation splits (train/val case IDs per fold)
from utils import split_dataset
print("\n step 3.4: split the dataset \n")
save_path = f"./dataset_csv_example/splits/5fold/{study}/"
os.makedirs(save_path, exist_ok=True)
split_dataset(clinical_new, save_path, fold=5)


 step 3.4: split the dataset 

the numbers of train case id in fold 0:  836
the numbers of val case id in fold 0:  210
the numbers of train case id in fold 1:  837
the numbers of val case id in fold 1:  209
the numbers of train case id in fold 2:  837
the numbers of val case id in fold 2:  209
the numbers of train case id in fold 3:  837
the numbers of val case id in fold 3:  209
the numbers of train case id in fold 4:  837
the numbers of val case id in fold 4:  209
